In [1]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re

In [ ]:

def get_vkids_ids(query=None, rango="0-4"):
    rangos = {"0-4": 0, "5-8": 1, "9-12": 2}
    driver = webdriver.Chrome()
    driver.get(f"https://www.youtubekids.com/search")
    try:
        #Selección del modo padre para poder configurar youtube kids con el rango de edad deseado
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "parent-button")))
        input_element = driver.find_element(By.ID, "parent-button")
        input_element.click()
        #Next para pasar a la siguiente pantalla
        WebDriverWait(driver, 1).until(EC.presence_of_element_located((By.ID, "next-button")))
        input_element = driver.find_element(By.ID, "next-button")
        input_element.click()
        year_digit_list = ['1','9','9','9']
        #Introducción del año de nacimiento para verificar la edad
        for i in range(4):
            WebDriverWait(driver, 1).until(EC.presence_of_element_located((By.ID, f"onboarding-age-gate-digit-{i+1}")))
            input_element = driver.find_element(By.ID, f"onboarding-age-gate-digit-{i+1}")
            input_element.clear()
            input_element.send_keys(year_digit_list[i])
        input_element.send_keys(Keys.ENTER)
        
        #Skip del video, porque si no hay que esperar 26 segundos para poder pasar a la siguiente pantalla
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "show-text-link")))
        input_element = driver.find_element(By.ID, "show-text-link")
        input_element.click()

        #Next para pasar a la siguiente pantalla, el problema es que se van acumulando los botones next, con el mismo id, 
        #pero solo uno es visible, por lo que hay que buscar el que está visible para poder hacer click en él.
        botones = driver.find_elements(By.ID, "next-button")

        boton_visible = [b for b in botones if b.is_displayed()][0]
        boton_visible.click()

        #Skip del tutorial
        WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.ID, "skip-button")))

        botones = driver.find_elements(By.ID, "skip-button")
        boton_visible = [b for b in botones if b.is_displayed()][0]
        boton_visible.click()

        #Next para pasar a la pantalla de selección ddel rango de edad
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "next-button")))

        botones = driver.find_elements(By.ID, "next-button")
        boton_visible = [b for b in botones if b.is_displayed()][0]
        boton_visible.click()

        # En esta parte hay 3 card container que es para seleccionar entre 3 rangos de edades diferentes
        # el primero es para niños menores de 5 años, el segundo para niños entre 6 y 8 años y el tercero para niños entre 9 y 12 años. Para este caso se selecciona el segundo rango de edad.
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "card-container")))
        buttons = driver.find_elements(By.ID, "card-container")
        buttons[rangos[rango]].click()

        #Aceptar el rango de edad seleccionado
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "select-link")))
        button = driver.find_element(By.ID, "select-link")
        button.click()

        #Permitir la búsqueda de videos
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "search-on-button")))
        button = driver.find_element(By.ID, "search-on-button")
        button.click()

        #Finalizar la configuración
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "done-button")))
        button = driver.find_element(By.ID, "done-button")
        button.click()

        #Si se desea realizar una búsqueda, se introduce la query en el buscador y se pulsa enter
        if query:
            WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "input")))
            input_element = driver.find_element(By.ID, "input")
            input_element.send_keys(query + Keys.ENTER)

        #Se actualiza la página ya que si no hay veces en las que no aparecen los vídeos en el html, y se espera 1 segundo a que cargue
        driver.refresh()
        time.sleep(1)
        # Se obtiene el código fuente de la página y se parsea con BeautifulSoup para obtener los ids de los vídeos
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        video_ids = []

        # Buscar todos los enlaces que contienen watch?v=
        links = soup.find_all("a", href=re.compile(r"watch\?v="))

        for link in links:
            href = link["href"]
            video_id = href.split("v=")[-1]
            video_ids.append(video_id)
        return video_ids
    
    finally:
        time.sleep(2)
        driver.quit()

In [3]:
kids_ids = get_vkids_ids()

In [4]:
kids_ids

['DIhKL8hbf4w',
 'LZeIiYBb8PI',
 'UBxqzZPA8SA',
 '60Ykwu4mfY4',
 'yrsUcALqXJ8',
 'YxVHj3CiGHg',
 'NiZ_GyrVSHU',
 'D-EnCCSxEUU',
 '0f3cQs2VDXY',
 'G4PUYnZPnCs',
 'ZgsjOJe4f58',
 'Gb98bJJDkAc',
 'YMtD2MsJ9Oc',
 '64msXB6DNTc',
 'lMGj-UC4Xqs',
 'YuQClVd3GaI',
 'Sq3--BrDxws',
 'T-uTC9vl-O4',
 'KfUYm6pyllQ',
 'jiR8Arh0x7o',
 'XRpgSvGYAn8',
 '2to_dDNLEyU',
 '6t0wy4vzEVo',
 'BZbr0sshwWI',
 'OG4oCCgHVJU',
 'IFsERoKHwec',
 'yRS1FIwUb7w',
 'WbPQW66ocF8',
 'CUCYYP9vSwk',
 'x1ziHhYYDd8',
 'fwHzex-EkpA',
 'HFTW5g8aU2I',
 '4YK8Wn0w5IU',
 'VV0fpDcGoqU',
 '7U5Elf6IB8Y',
 '5QNXO5MGBEA']